In [1]:
#Importing basic libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'numpy'

In [ ]:
#Importing google drive module to import data from Drive


In [ ]:
#Importing the datasets

oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
sample = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv')
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')

**Data Exploration and Cleaning**

In [2]:
def get_info(df):
  print(" -- DATA -- ")
  print(df.head())
  print(" -- Shape --")
  print(df.shape)
  print(" -- Feature datatypes and non null value details -- ")
  print(df.info())
  print(" -- Feature statistical details -- ")
  print(df.describe().transpose())
  print(" -- Columns -- ")
  print(df.columns)
  print(" -- Null Values -- ")
  print(df.isnull().sum())

In [ ]:
#OIL DATASET

get_info(oil)

In [ ]:
#Only contains missing values for one feature, imputing the missing with the previous valid observation

oil["dcoilwtico"].fillna(method = "bfill", inplace = True)

In [ ]:
#HOLIDAYS Dataset

get_info(holidays)

In [ ]:
#No missing values

#But in the dataset description, it is mentioned that the Transferred holidays must be taken special care of: 
holidays.loc[holidays.type=="Transfer", "description"] = holidays.loc[holidays.type == "Transfer", "description"].str.replace("Translado", "") ##Remove Translado word in the holidays as they are transferred holidays.
holidays['type'].replace("Transfer", "Normal", inplace = True) #Transferred holidays celebrated on that day from some other day

In [ ]:
# STORES Dataset

get_info(stores)

In [ ]:
# TRANSACTIONS DATASET

get_info(transactions)

In [ ]:
# TRAIN Dataset

get_info(train)

**Data Visualization**

In [ ]:
def date_form(df):
    df['date'] = pd.to_datetime(df['date'], format = "%Y-%m-%d")

In [ ]:
date_form(holidays)
date_form(oil)
date_form(train)
date_form(test)
date_form(transactions)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(10,5))
oil.plot.line(x="date", y="dcoilwtico", color="b", ax=axes, rot=0)
plt.title("Dependency of the oil from the data")
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(10,5))
transactions.plot.line(x="date", y="transactions", color="b", ax=axes, rot=0)
plt.title("No of transactions per date")
plt.show()

In [ ]:
def plot_stats(df, column, ax,color,angle):
    count_classes = df[column].value_counts()
    ax = sns.barplot(x=count_classes.index, y=count_classes, ax=ax, palette=color)
    ax.set_title(column.upper(), fontsize=20)
    for tick in ax.get_xticklabels():
        tick.set_rotation(angle)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15,5))
fig.autofmt_xdate()
fig.suptitle("Stats of df_holidays".upper())
plot_stats(holidays, "type", axes[0], "pastel", 45)
plot_stats(holidays, "locale", axes[1], "rocket", 45)
plt.show()

In [ ]:
#STORES

In [ ]:
fig, axes = plt.subplots(nrows = 4, ncols=1, figsize=(10, 15))
plot_stats(stores, "city", axes[0], "mako_r", 45)
plot_stats(stores, "state", axes[1], "rocket_r", 45)
plot_stats(stores, "type", axes[2], "magma", 0)
plot_stats(stores, "cluster", axes[3], "viridis", 0)

**Data merging starts**

In [ ]:
train.head()

In [ ]:
df = train.merge(holidays, on = 'date', how = 'left')
df.head()

In [ ]:
df = df.merge(oil, on = 'date', how = 'left')
df.head()

In [ ]:
df = df.merge(stores, on = 'store_nbr', how = 'left')
df.head()

In [ ]:
df = df.merge(transactions, on = ['date', 'store_nbr'], how = 'left')
df.head()

In [ ]:
df = df.rename(columns = {'type_x' : "holiday_type", "type_y" : "store_type"})
df.head()

In [ ]:
df['holiday_type'] = df['holiday_type'].replace({np.nan : "Normal"})

In [ ]:
#Removing rows having transactions MISSING

df = df[df["transactions"].isnull() == False]

In [ ]:
df["dcoilwtico"] = df["dcoilwtico"].fillna(method = 'bfill')

In [ ]:
df.head()

In [ ]:
#Dropping unneeded data

df = df.drop(['locale_name', "description", "transferred"], axis = 1)
df = df.drop(['locale', 'family', 'city', 'state', 'cluster', 'store_type'], axis = 1)

**Encoding Categorical Values**

In [ ]:
#One hot encoding the categorical values -- type_x (store type) and type_y (holiday type)

from sklearn.preprocessing import OneHotEncoder
low_card_cols = ["holiday_type"]
low_card_enc = OneHotEncoder(handle_unknown = "ignore", sparse = False)
low_card_df = pd.DataFrame(low_card_enc.fit_transform(df[low_card_cols])) # creating a seperate Dataframe to hold the encoded values
low_card_df.index = df.index #To make sure merging happens correctly
df_encoded = pd.concat([df.drop(low_card_cols, axis = 1), low_card_df], axis=1)

df_encoded.head()

**Dropping columns that wont be needed for prediction**

In [ ]:
df_encoded.drop('store_nbr', axis = 1, inplace = True)
df_encoded.drop('id', axis = 1, inplace = True)

In [ ]:
df_encoded.head()

In [ ]:
df_f = df_encoded.groupby("date").agg(np.mean) # Grouping the dataframe results into 1 row for each date and taking mean of all the other values and aggregating into one record
df_f.head()

In [ ]:
df_f.isnull().sum()  #Final dataset doesnt have any missing values, thats good! 

In [ ]:
df_f = df_f.astype('float64')  #Converting all features to float values

**Function to create dataframe with WINDOW = 1 and LAG = 1 for Time Series Analysis**

In [ ]:
def series_to_supervised2(data, n_in=1, n_out=1, dropnan=True):
    n_vars = 1 if type(data) is list else data.shape[1]
    df = pd.DataFrame(data)
    cols, names = list(), list()
    # input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
        names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
    # forecast sequence (t, t+1, ... t+n)
    for i in range(0, n_out):
        cols.append(df.shift(-i))
        if i == 0:
            names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
        else:
            names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
    # put it all together
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    # drop rows with NaN values
    if dropnan:
        agg.dropna(inplace=True)
    return agg

**Normalizing Data for faster LSTM fitting**

In [ ]:
values = df_f.values

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range = (0,1))

scaled_df_split = scaler.fit_transform(values)

In [ ]:
window = 1
lag = 1
series = series_to_supervised2(scaled_df_split, n_in=window, n_out=lag)

In [ ]:
series.head()

In [ ]:
series.describe().transpose()

In [ ]:
#These are the columns we dont want to predict, we drop them. We only have to predict sales and not the rest.
series.drop(series.columns[[11,12,13,14,15,16,17,18,19]], axis=1, inplace=True)

In [ ]:
series.head() # Final dataset that we pass to LSTM models.

In [ ]:
series_values = series.values

In [ ]:
# Label
labels = series["var1(t)"] # SALES VALUE - DEPENDENT VARIABLE
series = series.drop("var1(t)", axis=1) # INDEPENDENT VARIABLES

In [ ]:
l_values = labels.values
s_values = series.values

In [ ]:
series.shape

In [ ]:
split_length = 365*3 #Will train on 3 years of data and predict the rest

X_train = s_values[:split_length]
X_test = s_values[split_length:]

y_train = l_values[:split_length]
y_test = l_values[split_length:]

print("Train shape: ", X_train.shape, y_train.shape)
print("Test shape: ", X_test.shape, y_test.shape)

In [ ]:
# Reshape for LSTM input

X_train = X_train.reshape((X_train.shape[0], 1, 10))
X_test = X_test.reshape((X_test.shape[0],1, 10))

print("X_train size: ", X_train.shape)
print("X_test size: ", X_test.shape)

In [ ]:
y_train.shape

In [ ]:
from keras.optimizers import Adam

epochs = 100
batch = 128
lr = 0.0001
adam = Adam(lr)

In [ ]:
from keras.models import Sequential
from keras.layers import LSTM, Dropout, Dense 
from sklearn.metrics import mean_squared_error,r2_score



In [ ]:
print(X_train.shape[1], X_train.shape[2])

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Flatten

def create_model(optimizer='adam', neurons=32, dropout_rate=0.2):
    model = Sequential()

    # CNN layers
    model.add(Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(1, 10), padding='same'))


    # LSTM layer
    model.add(Bidirectional(LSTM(neurons, return_sequences=True)))
    model.add(Bidirectional(LSTM(neurons)))
    model.add(Dropout(dropout_rate))

    # Flatten layer
    model.add(Flatten())

    # Output layer
    model.add(Dense(1, activation='linear'))

    model.compile(optimizer=optimizer, loss='mse')  # Mean Squared Error as loss for regression task
    return model

In [ ]:
import tensorflow as tf
callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

In [ ]:
cv = []
for op in ['adam', 'rmsprop']:
    for ne in [32, 64]:
        for dr in [0.2, 0.3]:
            model = create_model(op, ne, dr)
            for epochs in [50, 100]:
                history = model.fit(X_train, y_train, validation_data = (X_test, y_test), epochs=epochs, callbacks=callback, verbose=1)
                di = {'optimizer': op, 'neurons': ne, 'dropout_rate': dr, 'epochs': epochs, 'loss': min(history.history['loss'])}
                cv.append(di)

results = pd.DataFrame.from_dict(cv)
results.sort_values(by='loss')

In [ ]:
results.sort_values(by='loss')

In [ ]:
model1 = create_model('adam', 32, 0.3)
history1 = model1.fit(X_train, y_train, validation_data = (X_test, y_test), epochs=epochs, callbacks=callback, verbose=1)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history1.history['loss'], label='train')
plt.plot(history1.history['val_loss'], label='validation')
plt.legend()
plt.show()

In [ ]:
yhat = model1.predict(X_test)
X_test = X_test.reshape((X_test.shape[0], X_test.shape[2]))
# invert scaling for forecast
inv_yhat = np.concatenate((yhat, X_test[:, 1:]), axis=1)
inv_yhat = scaler.inverse_transform(inv_yhat)
inv_yhat = inv_yhat[:,0]
# invert scaling for actual
y_test = y_test.reshape((len(y_test), 1))
inv_y = np.concatenate((y_test, X_test[:, 1:]), axis=1)
inv_y = scaler.inverse_transform(inv_y)
inv_y = inv_y[:,0]
# calculate RMSE
rmse = np.sqrt(mean_squared_error(inv_y, inv_yhat))
print('Test RMSE: %.12f' % rmse)

In [ ]:
model2 = create_model('adam', 64, 0.2)
history2 = model2.fit(X_train, y_train, validation_data = (X_test, y_test), epochs=epochs, callbacks=callback, verbose=1)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history2.history['loss'], label='train')
plt.plot(history2.history['val_loss'], label='validation')
plt.legend()
plt.show()

In [ ]:
yhat2 = model2.predict(X_test)
X_test = X_test.reshape((X_test.shape[0], X_test.shape[2]))
# invert scaling for forecast
inv_yhat2 = np.concatenate((yhat2, X_test[:, 1:]), axis=1)
inv_yhat2 = scaler.inverse_transform(inv_yhat2)
inv_yhat2 = inv_yhat2[:,0]
# invert scaling for actual
y_test = y_test.reshape((len(y_test), 1))
inv_y2 = np.concatenate((y_test, X_test[:, 1:]), axis=1)
inv_y2 = scaler.inverse_transform(inv_y2)
inv_y2 = inv_y2[:,0]
# calculate RMSE
rmse = np.sqrt(mean_squared_error(inv_y2, inv_yhat2))
print('Test RMSE: %.12f' % rmse)

**MAKING GRAPHS FOR ACTUAL vs TRAIN PREDICTION vs TEST PREDICTION**

LSTM - Model 1

In [ ]:
model1.summary()

In [ ]:
nsamples, nx, ny = X_train.shape
xtq = X_train.reshape((nsamples,nx*ny))

In [ ]:
train_predict=model.predict(X_train)

In [ ]:
# test_predict.shape

In [ ]:
# xt = np.concatenate((yhat, X_test[:, 1:]), axis=1)
# test_predict=scaler.inverse_transform(xt)


In [ ]:
# nsamples, nx, ny = X_test.shape
# test_predict = scaler.inverse_transform(xt)

In [ ]:
X_train = X_train.reshape((X_train.shape[0], X_train.shape[2]))
# invert scaling for forecast
train_predict = np.concatenate((train_predict, X_train[:, 1:]), axis=1)
train_predict = scaler.inverse_transform(X_train)

In [ ]:
train_predict.shape

In [ ]:
xt = np.concatenate((yhat, X_test[:, 1:]), axis=1)
test_predict=scaler.inverse_transform(xt)

In [ ]:

look_back=1
# trainPredictPlot = numpy.empty_like(df1)
trainPredictPlot = np.empty_like(s_values)

trainPredictPlot[:, :] = np.nan
trainPredictPlot[look_back:len(train_predict)+look_back, :] = train_predict
# shift test predictions for plotting
testPredictPlot = np.empty_like(s_values)
testPredictPlot[:, :] = np.nan
testPredictPlot[len(train_predict)+(look_back*2)-3:len(s_values)-1, :] = test_predict
# testPredictPlot[1200:] = test_predict

# plot baseline and predictions
plt.plot(scaler.inverse_transform(s_values), color = 'green', label='actual')
plt.plot(trainPredictPlot, color = 'blue', label='train_predictions')
plt.plot(testPredictPlot, color = 'red', label='test_predictions')
# plt.plot(test_predict)
plt.figure(figsize=(30,15))
plt.show()